# Liora Engine · prueba P0 real en GPU gratuita (0 €)

**Kaggle Notebooks** (gratis, sin tarjeta). Requisitos de la cuenta: teléfono verificado (activa GPU e Internet).
En el panel derecho: *Session options → Accelerator: **GPU T4 x2*** (o P100) · *Internet: **On***. Después: **Run all**.

La prueba P0 usa una **mujer ficticia generada solo con texto**: no se sube ninguna foto. Nada se simula: si falta GPU,
memoria o pesos, el motor se detiene y lo deja escrito en `evidence.json`.
Al terminar, descarga desde *Output*: `liora-smoke.mp4`, `evidence.json`, `run.log` y `capacity.json`.

In [ ]:
# 0) Detenerse ANTES de instalar o descargar nada si Kaggle no ha concedido GPU
import os, shutil, subprocess, pathlib
assert shutil.which('nvidia-smi'), 'No hay GPU NVIDIA asignada: Session options → Accelerator → GPU T4 x2, y reinicia.'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
assert gpu.returncode == 0 and gpu.stdout.strip(), 'GPU NVIDIA no utilizable en esta sesión.'
import torch
assert torch.cuda.is_available(), 'PyTorch no detecta CUDA: no se descargan los pesos.'

In [ ]:
import os, pathlib
WORK = pathlib.Path('/kaggle/working' if os.path.isdir('/kaggle/working') else 'liora_work').resolve()
WORK.mkdir(exist_ok=True)
# Pesos fuera de /kaggle/working (límite de 20 GB de salida); /root tiene más espacio en Kaggle
os.environ.setdefault('HF_HOME', '/root/.cache/huggingface')
!nvidia-smi
!free -g; df -h /root "$WORK" 

In [ ]:
# Código del motor: solo la carpeta liora/engine del repositorio público
!rm -rf edge-funding && git clone -q --depth 1 --filter=blob:none --sparse https://github.com/alvarogavilan/edge-funding.git
!cd edge-funding && git sparse-checkout set liora/engine && git log -1 --format='commit %H %cd'
os.chdir('edge-funding/liora/engine')
# torch ya viene instalado en Kaggle: solo se añaden dependencias del motor
!pip install -q "diffusers>=0.35" "transformers>=4.49" "accelerate>=1.1" ftfy sentencepiece imageio-ffmpeg "huggingface_hub>=0.30"
!python -c "import torch, diffusers; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'diffusers', diffusers.__version__)" 

In [ ]:
# 1) Capacidad real de la GPU gratuita
!python -m liora_engine probe --bench --out "$WORK/capacity.json" | head -40

In [ ]:
# 2) Pesos de Wan 2.2 TI2V-5B (Apache-2.0), con tiempo y tamaño medidos
import time, subprocess
from huggingface_hub import snapshot_download
t0 = time.time()
MODEL = snapshot_download('Wan-AI/Wan2.2-TI2V-5B-Diffusers')
size = int(subprocess.check_output(['du', '-sbL', MODEL]).split()[0])
print(MODEL, round(size / 1e9, 1), 'GB descargados en', round(time.time() - t0), 's')
os.environ['LIORA_WAN22_TI2V'] = MODEL

In [ ]:
# 3) PRUEBA P0: 3,5 s (85 fotogramas) a 704×1280, 30 pasos. Escribe liora-smoke.mp4 + evidence.json + run.log
!python -m liora_engine smoke --work "$WORK" --seconds 3.5 --steps 30

In [ ]:
# 3b) Solo si 3) marcó «fotogramas INVÁLIDOS» en una T4/P100 (fp16): repetir en bfloat16 (más lento)
import json
ev = json.load(open(WORK / 'evidence.json'))
print(ev['status'], ev.get('frame_health'))
if ev['status'] == 'invalid_output':
    os.environ['LIORA_DTYPE'] = 'bfloat16'
    !python -m liora_engine smoke --work "$WORK" --seconds 3.5 --steps 30

In [ ]:
# 4) Evidencia y reproducción
import json
from IPython.display import Video
ev = json.load(open(WORK / 'evidence.json'))
print(json.dumps({k: ev.get(k) for k in ('status', 'generated', 'generation_s', 'total_s', 'model_revision', 'params', 'mp4', 'frame_health')}, indent=2, ensure_ascii=False))
print(json.dumps(ev['hardware']['gpu'], indent=2))
Video(str(WORK / 'liora-smoke.mp4'), embed=True, width=360) if ev.get('generated') else print('Sin MP4 válido: ver error/preflight en evidence.json')

In [ ]:
# 5) Verificación independiente con ffprobe (no depende del motor)
import json, hashlib, subprocess
clip = WORK / 'liora-smoke.mp4'
assert clip.exists() and clip.stat().st_size > 10000, 'No existe un MP4 válido: la prueba P0 NO ha concluido.'
info = json.loads(subprocess.check_output(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-count_frames',
    '-show_entries', 'stream=codec_name,profile,pix_fmt,width,height,r_frame_rate,nb_read_frames,duration', '-of', 'json', str(clip)]))
print(json.dumps({'file': clip.name, 'bytes': clip.stat().st_size, 'sha256': hashlib.sha256(clip.read_bytes()).hexdigest(), **info['streams'][0]}, indent=2))

### Siguiente (opcional, privado): escenas con la foto oficial
Sube la foto oficial como **Dataset privado** y ejecuta las dos celdas siguientes. La foto nunca va al repositorio:
el paquete de identidad solo guarda descriptores y su huella SHA-256.

In [ ]:
!pip install -q dlib-bin face_recognition_models==0.3.0 mediapipe && (apt-get -qq install -y libegl1 libgles2 > /dev/null 2>&1 || true)
import glob
cands = [p for e in ('jpg', 'jpeg', 'png') for p in glob.glob(f'/kaggle/input/**/*.{e}', recursive=True)]
OFFICIAL = cands[0] if cands else None
print('Foto oficial:', OFFICIAL)
if OFFICIAL:
    !python -m liora_engine identity-enroll --pack "$WORK/identity-pack.json" --image "$OFFICIAL" --official
    !python -m liora_engine plan --official "$OFFICIAL" --pack "$WORK/identity-pack.json" --out "$WORK/job.json" --steps 30

In [ ]:
# Etapa 1 (y luego --only 2, 3… : cada etapa parte del último fotograma aceptado)
if OFFICIAL:
    !python -m liora_engine render --job "$WORK/job.json" --work "$WORK/render" --only 1
    !ls -la "$WORK/render"/scene-*